# Лабораторная работа 5. Диффузионные модели (Stable Diffusion)

В этой лабораторной работе мы дообучим модель **Stable Diffusion v1.5** с помощью метода **DreamBooth** и **LoRA**, чтобы она научилась генерировать изображения с конкретным человеком (мной).

**Цели:**
1. Настроить окружение и загрузить базовую модель.
2. Сгенерировать изображения класса (Prior Preservation) для предотвращения деградации модели.
3. Обучить LoRA-адаптер на 10-15 фотографиях пользователя.
4. Сгенерировать изображения в разных стилях (киберпанк, ведьмак и т.д.).
5. Оценить качество генераций с помощью метрик (Face Similarity, CLIP Score, BRISQUE).


## §1. Dev — Подготовка окружения
Клонирование или обновление репозитория (Git Pull).


In [ ]:
import os
import sys
from pathlib import Path

if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import userdata  # type: ignore
    github_token = userdata.get('GITHUB_TOKEN')
    repo_url = f"https://{github_token}@github.com/Ma-XD/ITMO-CV.git"
    
    if not Path("/content/ITMO-CV").exists():
        !git clone {repo_url} /content/ITMO-CV
    else:
        %cd /content/ITMO-CV
        !git pull
    
    sys.path.insert(0, "/content/ITMO-CV/lab5-DIF")
    %cd /content/ITMO-CV/lab5-DIF
else:
    LAB_DIR = Path(".").resolve()
    if LAB_DIR.name != "lab5-DIF":
        %cd lab5-DIF
    if str(LAB_DIR) not in sys.path:
        sys.path.insert(0, str(LAB_DIR))


## §2. Установка зависимостей
Монтирование Google Drive (в Colab) и установка библиотек.


In [ ]:
from env_config import is_colab

if is_colab:
    from google.colab import drive  # type: ignore
    # drive.mount идемпотентен: если уже примонтирован, он просто напишет 'Drive already mounted'
    drive.mount('/content/drive')
    
    # pip install идемпотентен: если пакеты уже установлены, он напишет 'Requirement already satisfied'
    !pip install -r requirements.txt


## §3. Импорты и проверка конфигурации
Загрузка путей из `config.py` и проверка доступности GPU.


In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Добавляем папку лабы в sys.path для импорта конфигов
lab_dir = Path("lab5-DIF").resolve() if not is_colab else Path("/content/ITMO-CV/lab5-DIF")
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

from env_config import print_env, get_device
from config import (
    INSTANCE_DIR, CLASS_DIR, CHECKPOINT_DIR, FIGURE_DIR,
    MODEL_ID, INSTANCE_PROMPT, CLASS_PROMPT, RESOLUTION
)

print_env()
device = get_device()

print(f"\nПапка с фото пользователя (Instance): {INSTANCE_DIR}")
print(f"Папка с фото класса (Class): {CLASS_DIR}")
print(f"Чекпоинты сохраняются в: {CHECKPOINT_DIR}")


## §4. Загрузка базовой модели
Загружаем оригинальную модель Stable Diffusion v1.5 (идемпотентно).


In [ ]:
from diffusers import StableDiffusionPipeline
import torch

try:
    if pipeline is not None:
        print("✅ Базовая модель уже находится в памяти.")
except NameError:
    print("Загрузка базовой модели SD 1.5...")
    pipeline = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
        safety_checker=None  # Отключаем safety checker
    )
    pipeline = pipeline.to(device)
    if device.type == "cuda":
        pipeline.enable_attention_slicing()
    print("✅ Базовая модель успешно загружена!")


### Smoke-test базовой модели
Генерируем тестовое изображение до дообучения, чтобы убедиться в работоспособности.


In [ ]:
# Smoke-test: генерация тестового изображения
prompt = "a photo of a cat"
generator = torch.Generator(device=device).manual_seed(42)

print(f"Генерируем: '{prompt}'...")
test_image = pipeline(prompt, generator=generator, num_inference_steps=30).images[0]

# Отрисовка результата
plt.figure(figsize=(6, 6))
plt.imshow(test_image)
plt.axis('off')
plt.title("Smoke-test: Базовая модель SD 1.5")
plt.show()


## §5. Генерация изображений класса (Prior Preservation)
Создаем 150 изображений обычных людей, чтобы модель не забыла их после дообучения на лице пользователя. Ячейка идемпотентна.


In [ ]:
import hashlib
from tqdm.auto import tqdm

num_class_images = 150
existing_images = list(CLASS_DIR.glob("*.jpg")) + list(CLASS_DIR.glob("*.png"))

if len(existing_images) >= num_class_images:
    print(f"✅ Изображения класса уже существуют ({len(existing_images)} шт.). Пропускаем генерацию.")
else:
    images_to_generate = num_class_images - len(existing_images)
    print(f"Сбор изображений класса. Нужно сгенерировать {images_to_generate} шт.")
    
    # Отключаем градиенты для ускорения генерации
    with torch.no_grad():
        for i in tqdm(range(images_to_generate), desc="Генерация class images"):
            # Генерируем изображение
            image = pipeline(CLASS_PROMPT, num_inference_steps=30).images[0]
            
            # Сохраняем с уникальным именем (хэш)
            hash_name = hashlib.sha1(image.tobytes()).hexdigest()[:10]
            image_path = CLASS_DIR / f"class_{hash_name}.jpg"
            image.save(image_path)
            
    print("✅ Генерация изображений класса завершена.")
